In [ ]:
pip install hidapi pygame

In [9]:
import hid

VENDOR_ID = 0x0A07
PRODUCT_ID = 200

class ADU200:
    def __init__(self):
        self.dev = hid.device()
        self.dev.open(VENDOR_ID, PRODUCT_ID)
        print("Connected to ADU200")

        # Clear stale reads
        while self.read(50) is not None:
            pass

    def write(self, cmd):
        packet = chr(0x01) + cmd + chr(0) * max(7 - len(cmd), 0)
        return self.dev.write(packet.encode())

    def read(self, timeout=200):
        data = self.dev.read(8, timeout)
        if not data:
            return None
        s = ''.join(chr(n) for n in data[1:])
        s = s.split('\x00', 1)[0]
        return s if s else None

    def close(self):
        self.dev.close()


    # Relay control
    def relay_set(self, n):
        return self.write(f"SK{n}")

    def relay_reset(self, n):
        return self.write(f"RK{n}")

    def relay_toggle(self, n):
        return self.write(f"TK{n}")

    def relay_read(self, n):
        self.write(f"RPK{n}")
        return self.read()
    # Read full PORT A (binary string)
    def read_portA(self):
        self.write("RPA")
        return self.read()

    # Read PORT A as integer
    def read_portA_int(self):
        self.write("RPAI")
        val = self.read()
        return int(val) if val is not None else None

    # Read a single input bit
    def read_input(self, n):
        self.write(f"RPA{n}")
        return self.read()


In [ ]:
adu = ADU200()

Connected to ADU200


In [ ]:
adu.relay_set(0)

In [ ]:
adu.relay_reset(0)

In [ ]:
state = adu.relay_read(0)
print("Relay 0 state:", state)

Relay 0 state: 0


In [ ]:
pa = adu.read_portA()
print("PORT A (binary):", pa)

PORT A (binary): 0000


In [ ]:
val = adu.read_portA_int()
print("PORT A (int):", val)

PORT A (int): None


In [ ]:
bit3 = adu.read_input(3)
print("Input 3:", bit3)

Input 3: 0


In [ ]:
adu.close()
# Optional

## This one creates the window and controls the relay at the same time!

In [2]:
import hid
import pygame

VENDOR_ID = 0x0A07
PRODUCT_ID = 200

class ADU200:
    def __init__(self):
        self.dev = hid.device()
        self.dev.open(VENDOR_ID, PRODUCT_ID)
        print("Connected to ADU200")

        # Clear stale reads
        while self.read(50) is not None:
            pass

    def write(self, cmd):
        packet = chr(0x01) + cmd + chr(0) * max(7 - len(cmd), 0)
        return self.dev.write(packet.encode())

    def read(self, timeout=200):
        data = self.dev.read(8, timeout)
        if not data:
            return None
        s = ''.join(chr(n) for n in data[1:])
        s = s.split('\x00', 1)[0]
        return s if s else None

    def close(self):
        self.dev.close()


    # Relay control
    def relay_set(self, n):
        return self.write(f"SK{n}")

    def relay_reset(self, n):
        return self.write(f"RK{n}")

    def relay_toggle(self, n):
        return self.write(f"TK{n}")

    def relay_read(self, n):
        self.write(f"RPK{n}")
        return self.read()
    # Read full PORT A (binary string)
    def read_portA(self):
        self.write("RPA")
        return self.read()

    # Read PORT A as integer
    def read_portA_int(self):
        self.write("RPAI")
        val = self.read()
        return int(val) if val is not None else None

    # Read a single input bit
    def read_input(self, n):
        self.write(f"RPA{n}")
        return self.read()

# ---------- PYGAME + JOYSTICK SETUP ----------

pygame.init()
pygame.joystick.init()

if pygame.joystick.get_count() == 0:
    raise RuntimeError("No joystick detected")

joy = pygame.joystick.Joystick(0)
joy.init()
print("Using joystick:", joy.get_name())

# Screen setup
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Joystick + ADU200 Demo")

clock = pygame.time.Clock()

# Center and geometry
cx, cy = WIDTH // 2, HEIGHT // 2
SQUARE_SIZE = 75
HALF_SQ = SQUARE_SIZE // 2
CIRCLE_RADIUS = 10
MAX_OFFSET = 100  # full deflection moves 100 px

# ADU200 instance
adu = ADU200()
RELAY_NUM = 0  # which relay to click

# Track inside/outside state to avoid spamming clicks
was_inside = True

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Read joystick axes
    ax_x = joy.get_axis(0)  # -1 (left) to +1 (right)
    ax_y = joy.get_axis(1)  # -1 (up) to +1 (down)

    # Map to pixel offsets
    offset_x = int(ax_x * MAX_OFFSET)
    offset_y = int(ax_y * MAX_OFFSET)

    circle_x = cx + offset_x
    circle_y = cy + offset_y

    # Check if circle center is inside the white square
    left = cx - HALF_SQ
    right = cx + HALF_SQ
    top = cy - HALF_SQ
    bottom = cy + HALF_SQ

    inside = (left <= circle_x <= right) and (top <= circle_y <= bottom)

    # Detect transition from inside -> outside and click relay
    if was_inside and not inside:
        print("Circle left square -> relay click")
        adu.relay_set(0)
    elif was_inside:
        adu.relay_reset(0)

    was_inside = inside

    # Draw
    screen.fill((0, 128, 0))  # green background

    # White square
    pygame.draw.rect(
        screen,
        (255, 255, 255),
        (cx - HALF_SQ, cy - HALF_SQ, SQUARE_SIZE, SQUARE_SIZE),
        0
    )

    # Red circle
    pygame.draw.circle(screen, (255, 0, 0), (circle_x, circle_y), CIRCLE_RADIUS)

    pygame.display.flip()
    clock.tick(60)

adu.close()
pygame.quit()


Using joystick: Controller (Gamepad F310)
Connected to ADU200
Circle left square -> relay click
Circle left square -> relay click
Circle left square -> relay click
Circle left square -> relay click
Circle left square -> relay click
Circle left square -> relay click
Circle left square -> relay click
Circle left square -> relay click
Circle left square -> relay click
